# Qwen2-VL Anomaly Explanation Pipeline

Standalone notebook — reads pre-extracted frames and metadata written by
`full_pipeline_colab.ipynb` (Steps 1-5) from Google Drive.  
No re-extraction, no RTFM re-run.

### Prerequisites on Drive
```
MyDrive/
  rtfm_pipeline_outputs/          ← written by full_pipeline_colab
    01_0015/
      metadata.json               ← frame list + segment scores
      snippet_000_frame_0012.jpg  ← extracted frames
      ...
    ...
  annotations.json                ← human ground-truth explanations
```

### Steps in this notebook
1. GPU check + Drive mount
2. Install Qwen2-VL deps (run once, then **Runtime → Restart session**)
3. Load Qwen2-VL model
4. Build prompts from saved metadata + run inference
5. Judge with GPT-4o
6. Head-to-head comparison vs GPT-4o

In [1]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-80GB
Memory  : 85.1 GB


In [2]:
# ── Cell 2: Mount Drive + load metadata ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import torch
from pathlib import Path

OUTPUT_DIR       = Path('/content/drive/MyDrive/rtfm_pipeline_outputs')
ANNOTATIONS_PATH = Path('/content/drive/MyDrive/annotations.json')

# Load all per-video metadata written by full_pipeline_colab
video_metas = {}
for meta_file in sorted(OUTPUT_DIR.glob('*/metadata.json')):
    with open(meta_file) as f:
        meta = json.load(f)
    vname = meta['video_id']
    frames_with_files = [fr for fr in meta['extracted_frames'] if fr.get('file')]
    if not frames_with_files:
        continue
    # Attach resolved paths
    vid_dir = OUTPUT_DIR / vname
    for fr in frames_with_files:
        fr['_path'] = str(vid_dir / fr['file'])
    meta['extracted_frames'] = frames_with_files
    video_metas[vname] = meta

print(f'Found metadata for {len(video_metas)} videos in {OUTPUT_DIR}')
print(f'Examples: {list(video_metas)[:5]}')

# Load annotations
if ANNOTATIONS_PATH.exists():
    with open(ANNOTATIONS_PATH) as f:
        annotations_raw = json.load(f)
    annotations = {e['video_id']: e for e in annotations_raw if 'video_id' in e}
    print(f'Annotations: {len(annotations)} videos')
else:
    print('WARNING: annotations.json not found — judge step will be skipped')
    annotations = {}

# Load existing GPT-4o explanations for comparison (optional)
gpt4o_explanations = {}
for expl_file in OUTPUT_DIR.glob('*/explanation.json'):
    with open(expl_file) as f:
        d = json.load(f)
    gpt4o_explanations[d['video_id']] = d
print(f'GPT-4o explanations available: {len(gpt4o_explanations)}')
# ── Copy frames from Drive to local disk (avoids Drive FUSE I/O errors) ──────
import shutil
LOCAL_FRAMES = '/content/qwen_frames'
os.makedirs(LOCAL_FRAMES, exist_ok=True)

n_copied = 0
for vname, meta in video_metas.items():
    local_dir = os.path.join(LOCAL_FRAMES, vname)
    os.makedirs(local_dir, exist_ok=True)
    for fr in meta['extracted_frames']:
        drive_path = fr['_path']
        local_path = os.path.join(local_dir, fr['file'])
        if not os.path.exists(local_path):
            try:
                shutil.copy2(drive_path, local_path)
                n_copied += 1
            except Exception as e:
                print(f'  WARN: could not copy {drive_path}: {e}')
        fr['_path'] = local_path   # redirect to local copy

print(f'Frames cached locally: {n_copied} new files -> {LOCAL_FRAMES}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found metadata for 34 videos in /content/drive/MyDrive/rtfm_pipeline_outputs
Examples: ['01_0015', '01_0025', '01_0027', '01_0028', '01_0052']
Annotations: 44 videos
GPT-4o explanations available: 0


In [3]:
# ── Cell 3: Install Qwen2-VL dependencies ────────────────────────────────────
# Run this cell ONCE, then:  Runtime → Restart session
# After restart, re-run from Cell 1.
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'transformers>=4.45.0', 'accelerate', 'qwen-vl-utils[decord]', 'bitsandbytes',
])

import transformers
print(f'transformers : {transformers.__version__}')
print()
print('Install done.')
print('>>> Runtime → Restart session, then re-run from Cell 1.')

transformers : 5.3.0

Install done.
>>> Runtime → Restart session, then re-run from Cell 1.


In [ ]:
# ── Cell 4: Load Qwen-VL model ────────────────────────────────────────────────
#
# A100-80 GB options (pick ONE):
#
#   Qwen/Qwen3-VL-32B-Instruct   BF16  ~64 GB  ← DEFAULT (best quality/fit)
#   Qwen/Qwen2.5-VL-7B-Instruct  BF16  ~15 GB  ← fast, lots of headroom
#   Qwen/Qwen2-VL-72B-Instruct   4-bit ~38 GB  ← 4-bit + CPU offload
#
import os, gc, torch
from huggingface_hub import login
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

# ── HuggingFace auth ──────────────────────────────────────────────────────────
HF_TOKEN_MANUAL = os.environ.get('HF_TOKEN', '')

hf_token = HF_TOKEN_MANUAL or os.environ.get('HF_TOKEN', '')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print(f'HF authenticated.  ({hf_token[:8]}...{hf_token[-4:]})')
else:
    print('WARNING: No HF_TOKEN — downloads may be throttled.')

# ── Model selection ───────────────────────────────────────────────────────────
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-32B-Instruct'
# QWEN_MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
# QWEN_MODEL_ID = 'Qwen/Qwen2-VL-72B-Instruct'

# ── Import model class (Qwen3-VL if available, else fall back to Qwen2-VL) ───
try:
    from transformers import Qwen3VLForConditionalGeneration as QwenVLModel
    print('Using Qwen3VLForConditionalGeneration')
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLModel
    print('Falling back to Qwen2VLForConditionalGeneration')

# ── Free GPU memory before load ───────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free before load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# ── Build load kwargs ─────────────────────────────────────────────────────────
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

if '72B' in QWEN_MODEL_ID:
    from transformers import BitsAndBytesConfig
    load_kwargs = dict(
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        ),
        device_map='auto',
        max_memory={0: '70GiB', 'cpu': '48GiB'},
    )
else:
    # 32B BF16 = ~64 GB  |  7B BF16 = ~15 GB  — both fit on A100-80 GB
    load_kwargs = dict(
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

print(f'Loading {QWEN_MODEL_ID} ...')
qwen_model = QwenVLModel.from_pretrained(QWEN_MODEL_ID, **load_kwargs)

# Cap visual tokens per image: 256*28*28 ~200k px -> ~256 tokens/image
# Raise max_pixels toward 1280*28*28 if you have headroom after loading
qwen_processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

print('Model loaded.')
print(f'GPU allocated : {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'GPU reserved  : {torch.cuda.memory_reserved()/1e9:.1f} GB')


In [ ]:
# ── Cell 5: Inference helpers ─────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a surveillance video anomaly analyst. You will be shown a set of \
frames sampled from a surveillance video that has been flagged as anomalous \
by a weakly-supervised anomaly detection model (RTFM).

The frames are ordered temporally. Each frame comes from a specific temporal \
snippet of the video, and you are given the anomaly score for that snippet \
(0 = normal, 1 = highly anomalous).

The frames were specifically selected from the anomalous portions of the \
video — they represent the onset, peak, and resolution of the detected anomaly.

Your task: Based on ALL the frames and their anomaly scores together, provide \
a single concise explanation (2-3 sentences) of what anomalous activity is \
happening. Focus on:
- WHAT is happening (the specific anomalous activity)
- WHO/WHAT is involved (people, vehicles, objects — describe appearance)
- WHEN in the sequence it starts and ends
- WHY it is anomalous (how it deviates from normal pedestrian behaviour)

Respond with ONLY a JSON object in this exact format:
{\"explanation\": \"...\"}"""


def build_qwen_message(meta):
    """Build a Qwen2-VL chat message from saved metadata."""
    frames = meta['extracted_frames']
    segs   = meta['anomalous_segments']

    seg_str   = ', '.join(f"snippets {s['start']}-{s['end']}" for s in segs)
    score_str = ', '.join(
        f"snippet {f['snippet_idx']}={f['score']:.3f}" for f in frames
    )
    intro = (
        f"This video has {meta['n_segments']} temporal snippets (~16 frames each). "
        f"Anomalous segments: [{seg_str}]. "
        f"Video-level gate score: {meta['gate_score']:.3f}. "
        f"Below are {len(frames)} frames from the anomalous segments. "
        f"Per-snippet scores: [{score_str}]."
    )

    content = [{"type": "text", "text": intro}]
    for fr in frames:
        img_path = fr.get('_path') or str(OUTPUT_DIR / meta['video_id'] / fr['file'])
        if not Path(img_path).exists():
            continue
        content.append({
            "type": "text",
            "text": (f"Frame from snippet {fr['snippet_idx']} "
                     f"(frame #{fr['frame_num']}, score: {fr['score']:.3f}):")
        })
        content.append({"type": "image", "image": f"file://{img_path}"})

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": content},
    ]


def query_qwen(messages, max_new_tokens=300):
    """Run Qwen2-VL inference and return parsed JSON dict."""
    text_input = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    pvi = process_vision_info(messages)
    image_inputs, video_inputs = pvi[0], pvi[1]

    inputs = qwen_processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to('cuda')

    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_text = qwen_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    try:
        if output_text.startswith('```'):
            output_text = output_text.split('```')[1]
            if output_text.startswith('json'):
                output_text = output_text[4:]
        return json.loads(output_text)
    except json.JSONDecodeError:
        return {"explanation": output_text}


print('Inference helpers defined.')

In [ ]:
# ── Cell 6: Run Qwen2-VL on all videos ───────────────────────────────────────

qwen_explanations = {}
n_total = len(video_metas)

for i, (vname, meta) in enumerate(sorted(video_metas.items())):
    n_frames = len(meta['extracted_frames'])
    print(f'[{i+1}/{n_total}] {vname}  '
          f'gate={meta["gate_score"]:.3f}  frames={n_frames} ...', end=' ', flush=True)

    try:
        messages    = build_qwen_message(meta)
        response    = query_qwen(messages)
        explanation = response.get('explanation', '')
    except Exception as e:
        print(f'ERROR: {e}')
        explanation = f'ERROR: {e}'

    short = f'"{explanation[:80]}..."' if len(explanation) > 80 else f'"{explanation}"'
    print(f'-> {short}')

    result = {
        'video_id':      vname,
        'model':         QWEN_MODEL_ID,
        'gate_score':    meta['gate_score'],
        'n_frames_sent': n_frames,
        'explanation':   explanation,
    }
    qwen_explanations[vname] = result

    vid_out = OUTPUT_DIR / vname
    vid_out.mkdir(parents=True, exist_ok=True)
    with open(vid_out / 'explanation_qwen.json', 'w') as f:
        json.dump(result, f, indent=2)

with open(OUTPUT_DIR / 'qwen_explanations_summary.json', 'w') as f:
    json.dump(list(qwen_explanations.values()), f, indent=2)

print(f'\nDone. {len(qwen_explanations)} explanations saved.')
print(f'Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')

In [ ]:
# ── Cell 7: Judge Qwen2-VL explanations (GPT-4o as judge) ────────────────────
!pip install -q openai
from openai import OpenAI
import base64

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        print('API key loaded from Colab Secrets.')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    raise ValueError(
        'No OpenAI API key.\n'
        'Add OPENAI_API_KEY via the key icon in the Colab left sidebar.'
    )

os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
print(f'OpenAI client ready.  ({api_key[:8]}...{api_key[-4:]})')

NORMAL_GT   = 'There is nothing anomalous in this video. All pedestrians are walking normally.'
JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI-generated \
explanation of an anomalous event in a surveillance video.

You will be given:
1. A HUMAN ground-truth explanation written by someone who watched the full video.
2. An AI-GENERATED explanation produced by a vision-language model that only \
saw a few sampled frames guided by anomaly scores from a weakly-supervised model.

Score the AI explanation on these 4 criteria (each 1-5):
- correctness: Does the AI identify the same anomaly as the human?
- specificity: Does the AI mention specific details?
- completeness: Does the AI capture all aspects the human mentioned?
- fluency: Is the AI explanation well-written and clear?

Respond with ONLY a JSON object:
{\"correctness\": 1-5, \"specificity\": 1-5, \"completeness\": 1-5, \"fluency\": 1-5, \"justification\": \"...\"}"""


def query_judge(human_expl, ai_expl):
    user_msg = (
        f'HUMAN ground-truth explanation:\n"{human_expl}"\n\n'
        f'AI-generated explanation:\n"{ai_expl}"'
    )
    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            messages=[
                {'role': 'system', 'content': JUDGE_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            max_tokens=300,
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {'correctness': None, 'specificity': None,
                'completeness': None, 'fluency': None,
                'justification': f'ERROR: {e}'}


qwen_judge_results = []
n_anom, n_fp = 0, 0

for i, (vname, expl_data) in enumerate(sorted(qwen_explanations.items())):
    ai_expl = expl_data.get('explanation', '')
    if not ai_expl or ai_expl.startswith('ERROR'):
        continue

    clip_id = vname.split('_')[1] if '_' in vname else vname
    is_truly_anomalous = len(clip_id) == 4

    if is_truly_anomalous:
        human_data = annotations.get(vname)
        if not human_data:
            print(f'  SKIP {vname}: no annotation')
            continue
        human_expl = human_data['explanation']
        gt_start   = human_data.get('anomaly_start_frame')
        gt_end     = human_data.get('anomaly_end_frame')
        video_type = 'anomalous'
        n_anom += 1
    else:
        human_expl = NORMAL_GT
        gt_start = gt_end = None
        video_type = 'normal_FP'
        n_fp += 1

    print(f'[{i+1}] {vname}  [{video_type}]')
    print(f'  AI : "{ai_expl[:90]}..."' if len(ai_expl) > 90 else f'  AI : "{ai_expl}"')
    print(f'  GT : "{human_expl[:90]}..."' if len(human_expl) > 90 else f'  GT : "{human_expl}"')

    scores = query_judge(human_expl, ai_expl)
    print(f'  -> C={scores.get("correctness")} S={scores.get("specificity")} '
          f'Co={scores.get("completeness")} F={scores.get("fluency")}')
    print()

    result = {
        'video_id':          vname,
        'model':             QWEN_MODEL_ID,
        'video_type':        video_type,
        'human_explanation': human_expl,
        'ai_explanation':    ai_expl,
        'gate_score':        expl_data.get('gate_score'),
        'gt_anomaly_start':  gt_start,
        'gt_anomaly_end':    gt_end,
        'scores':            scores,
    }
    qwen_judge_results.append(result)

    vid_out = OUTPUT_DIR / vname
    with open(vid_out / 'judge_qwen.json', 'w') as f:
        json.dump(result, f, indent=2)

    time.sleep(0.3)

with open(OUTPUT_DIR / 'qwen_judge_summary.json', 'w') as f:
    json.dump(qwen_judge_results, f, indent=2)

print(f'Judged {len(qwen_judge_results)} videos  '
      f'({n_anom} anomalous + {n_fp} normal FP)')

In [ ]:
# ── Cell 8: Head-to-head comparison — GPT-4o vs Qwen2-VL ─────────────────────
import numpy as np

METRICS = ['correctness', 'specificity', 'completeness', 'fluency']


def summarise(judge_list, label):
    anom = [r for r in judge_list if r['video_type'] == 'anomalous']
    rows = {
        m: [r['scores'].get(m) for r in anom if r['scores'].get(m) is not None]
        for m in METRICS
    }
    print(f'\n{"─"*62}')
    print(f'  {label}  (anomalous n={len(anom)})')
    print(f'{"─"*62}')
    print(f'  {"Metric":<16s} {"Mean":>6s} {"Std":>6s} {"Min":>5s} {"Max":>5s}')
    print(f'  {"─"*46}')
    overall = []
    for m in METRICS:
        v = np.array(rows[m])
        overall.extend(rows[m])
        print(f'  {m:<16s} {v.mean():>6.2f} {v.std():>6.2f} '
              f'{v.min():>5.1f} {v.max():>5.1f}')
    ov = np.array(overall)
    print(f'  {"─"*46}')
    print(f'  {"OVERALL":<16s} {ov.mean():>6.2f} {ov.std():>6.2f} '
          f'{ov.min():>5.1f} {ov.max():>5.1f}')
    return {m: float(np.mean(rows[m])) for m in METRICS}


# Load GPT-4o judge results from Drive
gpt4o_judge = []
for jf in OUTPUT_DIR.glob('*/judge.json'):
    with open(jf) as f:
        gpt4o_judge.append(json.load(f))

print(f'GPT-4o judge results loaded: {len(gpt4o_judge)}')

gpt_stats  = summarise(gpt4o_judge,       f'GPT-4o')
qwen_stats = summarise(qwen_judge_results, f'Qwen2-VL  ({QWEN_MODEL_ID.split("/")[-1]})')

# Per-video delta table
qwen_map = {r['video_id']: r for r in qwen_judge_results if r['video_type'] == 'anomalous'}
gpt_map  = {r['video_id']: r for r in gpt4o_judge        if r['video_type'] == 'anomalous'}

print(f'\n{"="*75}')
print(f'  Per-video delta (Qwen − GPT-4o) on anomalous videos')
print(f'{"="*75}')
print(f'  {"Video":<14s} {"ΔC":>5s} {"ΔS":>5s} {"ΔCo":>5s} {"ΔF":>5s}  {"Avg Δ":>7s}')
print(f'  {"─"*55}')

deltas = []
for vname in sorted(qwen_map):
    if vname not in gpt_map:
        continue
    q = qwen_map[vname]['scores']
    g = gpt_map[vname]['scores']
    d = {m: (q.get(m) or 0) - (g.get(m) or 0) for m in METRICS}
    avg_d = np.mean(list(d.values()))
    deltas.append(avg_d)
    sign = '+' if avg_d >= 0 else ''
    print(f'  {vname:<14s} '
          f'{d["correctness"]:>+5.0f} {d["specificity"]:>+5.0f} '
          f'{d["completeness"]:>+5.0f} {d["fluency"]:>+5.0f}  '
          f'{sign}{avg_d:>6.2f}')

if deltas:
    mean_d = np.mean(deltas)
    sign = '+' if mean_d >= 0 else ''
    print(f'  {"─"*55}')
    print(f'  {"MEAN DELTA":<14s}{"":>22s}  {sign}{mean_d:>6.2f}')
print(f'{"="*75}')
print(f'\n  GPT-4o   overall mean : {np.mean(list(gpt_stats.values())):.2f}')
print(f'  Qwen2-VL overall mean : {np.mean(list(qwen_stats.values())):.2f}')